# Distribution & Separability Plots

Figures are driven from here; the QC step (run_qc.py) writes metric_summary.csv for the pipeline.

- Separability uses the raw merged.csv
- Model agreement uses merged_normalized.csv

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from config.datasets import cfg
from config.paths import RAW_DATA_DIR, DISTRIBUTIONS_DIR, METRIC_YAML
from config.analysis import COLORS_MAP

from analysis.distributions import (
    get_numeric_metrics,
    add_binder_class,
    make_long_scores,
    get_model_groups,
    load_metric_metadata,
    enrich_long_scores,
    plot_metric_distribution,
    plot_metric_separability,
    plot_metric_separability_single,
    plot_model_agreement,
)
# Uniform figure styling: one validated palette and recessive axes/grid for every figure
# in the project (config/palette.py). Call once, before plotting.
from config.palette import apply_plot_style
apply_plot_style()
# Auto-save every figure below to data/figures/<dataset>/ at print resolution, named after the
# plot and its key arguments (e.g. top_metrics_bar_pr_auc.png). An explicit save_path= still wins.
# Comment this out to go back to display-only.
from config.paths import FIGURES_DIR
from analysis.figures import set_figure_dir
set_figure_dir(FIGURES_DIR / cfg.name)

In [ ]:
base = RAW_DATA_DIR / cfg.name

# every figure now lands under data/figures/<dataset>/ (set_figure_dir above); the few
# multi-figure plots take it as output_dir
output_dir = FIGURES_DIR / cfg.name

print(cfg.name)

## Benchmark structure (Results 3.1)

Lineage groups are the unit of all grouped cross-validation, so the effective sample size is the number of GROUPS, not the number of samples. The fold numbers are exact and few, so they are reported as a table (`cv_folds.csv`, rendered into `section_3_1.md`); the group-size distribution is the part that needs a figure.

In [ ]:
from analysis.distributions import benchmark_structure, plot_benchmark_structure

groups, folds = benchmark_structure(pd.read_csv(base / "merged.csv"))
display(folds)

# folds=None -> the lineage panel alone (recommended for the thesis); pass `folds` for both panels
plot_benchmark_structure(groups, folds=None, show=True);

## Separability (raw)

In [ ]:
df = add_binder_class(pd.read_csv(base / "merged.csv"))
metrics = get_numeric_metrics(df)
model_groups = get_model_groups(metrics)
long_scores_df = make_long_scores(df, metrics)

print(df[["binder", "type", "binder_class"]].value_counts())
print(df["binder"].unique())
print(df["type"].unique())

plot_metric_separability(long_scores_df, model_groups, output_dir, COLORS_MAP, show=True)

## Model agreement (scaled)

In [ ]:
scaled_df = add_binder_class(pd.read_csv(base / "merged.csv"))
scaled_metrics = get_numeric_metrics(scaled_df)

scaled_long = make_long_scores(scaled_df, scaled_metrics)
scaled_long = enrich_long_scores(scaled_long, load_metric_metadata(METRIC_YAML))

plot_model_agreement(scaled_long, output_dir, show=True)

## Single metric (inline)

Two views of ONE metric, chosen by name -- the grid above is for scanning, these are for looking:

- `plot_metric_distribution` -- histogram by binder label
- `plot_metric_separability_single` -- strip over box per binder class, split by `source`
  (`hue=None` for one undivided box per class). Classes absent from the data are dropped, and the
  tick labels carry the per-class n, since the classes differ by an order of magnitude in size.

In [ ]:
# quick inline look at one metric's distribution by binder
plot_metric_distribution(df, metrics[10])

In [ ]:
# same construction as the grid above, one metric at full size
metric = "af3_iptm"      # <- pick any entry of `metrics`
plot_metric_separability_single(long_scores_df, metric, colors_map=COLORS_MAP, show=True)